# 02 — Feature Engineering
Build the complete feature matrix: wavelet denoising, technical indicators,
volatility microstructure, order flow, Smart Money Concepts, and MTF alignment.

In [ ]:
# Install dependencies
# torch, numpy, pandas are pre-installed by Colab — do NOT reinstall them.
# Pinning versions fights Colab's environment and causes resolver conflicts.
!pip install -q xgboost ccxt PyWavelets hmmlearn numba scikit-learn pyyaml \
    tqdm pyarrow

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys

# Clone/update repo from GitHub (local Colab filesystem — fast)
REPO_DIR = '/content/scalp2_repo'
if os.path.exists(os.path.join(REPO_DIR, '.git')):
    !git -C {REPO_DIR} pull --ff-only
else:
    !git clone https://github.com/umutergul74/Scalp2.git {REPO_DIR}

if not os.path.exists(os.path.join(REPO_DIR, 'scalp2', '__init__.py')):
    raise FileNotFoundError('scalp2 package not found after clone!')

sys.path.insert(0, REPO_DIR)

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s: %(message)s')

from scalp2.config import load_config
config = load_config(f'{REPO_DIR}/config.yaml')
config.data.processed_dir = '/content/drive/MyDrive/scalp2/data/processed'

In [ ]:
import pandas as pd

# Load cleaned data (1H primary, 4H MTF)
df_1h = pd.read_parquet(f'{config.data.processed_dir}/BTC_USDT_1h_clean.parquet')
df_4h = pd.read_parquet(f'{config.data.processed_dir}/BTC_USDT_4h_clean.parquet')

print(f'1h (primary): {len(df_1h)} bars')
print(f'4h (MTF):     {len(df_4h)} bars')


In [ ]:
from scalp2.features.builder import build_features, drop_warmup_nans, get_feature_columns

# Build features for each timeframe
print('Building 1h features...')
df_1h_feat = build_features(df_1h, config.features)

print('Building 4h features...')
df_4h_feat = build_features(df_4h, config.features)

print(f'1h features: {len(get_feature_columns(df_1h_feat))} columns')


In [ ]:
from scalp2.data.mtf_builder import build_mtf_dataset

# Align 4H MTF features onto 1H primary index
df_full = build_mtf_dataset(df_1h_feat, df_4h_feat, htf_labels=['4h'])
print(f'Full dataset: {len(df_full)} rows x {len(df_full.columns)} columns')

# Drop warmup NaNs
df_full = drop_warmup_nans(df_full)
print(f'After warmup removal: {len(df_full)} rows x {len(df_full.columns)} columns')

feature_cols = get_feature_columns(df_full)
print(f'Model features: {len(feature_cols)}')
print(f'Memory: {df_full.memory_usage(deep=True).sum() / 1e6:.1f} MB')


In [ ]:
# Save feature matrix
output_path = f'{config.data.processed_dir}/BTC_USDT_features.parquet'
df_full.to_parquet(output_path)
print(f'Saved feature matrix to {output_path}')

# Save feature column list
import json
with open(f'{config.data.processed_dir}/feature_columns.json', 'w') as f:
    json.dump(feature_cols, f)
print(f'Saved {len(feature_cols)} feature column names')